# Module 21 — A/B Testing for UI Changes: Conversion-Rate Analysis

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scipy, Statsmodels, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 6 (Statistical Tests for A/B Testing)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic A/B Test Data](#3-synthetic-ab-test-data)
4. [Statistical Testing](#4-statistical-testing)
5. [Bayesian A/B Testing](#5-bayesian-ab-testing)
6. [Minimum Detectable Effect](#6-minimum-detectable-effect)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why A/B testing matters for fintech UI?

A/B testing is **essential** for data-driven product decisions:
- **Conversion optimization**: Test UI changes to increase feature adoption.
- **Risk mitigation**: Validate changes before full rollout.
- **Causal inference**: Establish true lift, not just correlation.

**Common fintech experiments**:
1. **Button placement**: Test "Send Money" button position.
2. **Copy changes**: Test different call-to-action text.
3. **Flow variations**: Test simplified vs. detailed flows.
4. **Pricing experiments**: Test fee structures.

**Statistical approaches**:
- **Frequentist**: Two-proportion z-test, chi-squared test.
- **Bayesian**: Beta-Binomial model, probability of being best.
- **Power analysis**: Calculate required sample size upfront.

In this notebook we will:
1. Generate synthetic A/B test data for a "Send Money" button.
2. Run frequentist (z-test) and Bayesian analyses.
3. Calculate minimum detectable effect and power.
4. Visualize results for stakeholder communication.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Statistical libraries ────────────────────────────────────────
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic A/B Test Data

In [ ]:
# Generate synthetic A/B test data for "Send Money" button
N_USERS = 50000
CONVERSION_RATE_CONTROL = 0.12  # 12% baseline conversion
LIFT_TREATMENT = 0.03  # 3% absolute lift

# Split users 50/50
users = pd.DataFrame({
    'user_id': range(N_USERS),
    'group': np.random.choice(['control', 'treatment'], N_USERS, p=[0.5, 0.5])
})

# Generate conversions
users['converted'] = 0
users.loc[users['group'] == 'control', 'converted'] = np.random.binomial(
    1, CONVERSION_RATE_CONTROL, (users['group'] == 'control').sum()
)
users.loc[users['group'] == 'treatment', 'converted'] = np.random.binomial(
    1, CONVERSION_RATE_CONTROL + LIFT_TREATMENT, (users['group'] == 'treatment').sum()
)

# Summary
summary = users.groupby('group')['converted'].agg(['sum', 'count', 'mean'])
summary.columns = ['conversions', 'users', 'conversion_rate']

print("=" * 70)
print("A/B TEST DATA SUMMARY")
print("=" * 70)
print(summary)
print(f"\nObserved lift: {(summary.loc['treatment', 'conversion_rate'] - summary.loc['control', 'conversion_rate'])*100:.2f}%")

---
## §4 · Statistical Testing

In [ ]:
# Two-proportion z-test
control = users[users['group'] == 'control']
treatment = users[users['group'] == 'treatment']

# Counts
n_control = len(control)
n_treatment = len(treatment)
x_control = control['converted'].sum()
x_treatment = treatment['converted'].sum()

# Z-test
z_stat, p_value = proportions_ztest(
    [x_control, x_treatment],
    [n_control, n_treatment],
    alternative='two-sided'
)

# Confidence intervals
ci_control = proportion_confint(x_control, n_control, alpha=0.05, method='wilson')
ci_treatment = proportion_confint(x_treatment, n_treatment, alpha=0.05, method='wilson')

print("=" * 70)
print("TWO-PROPORTION Z-TEST RESULTS")
print("=" * 70)
print(f"\nControl group:")
print(f"  Conversion rate: {x_control/n_control*100:.2f}%")
print(f"  95% CI: [{ci_control[0]*100:.2f}%, {ci_control[1]*100:.2f}%]")
print(f"\nTreatment group:")
print(f"  Conversion rate: {x_treatment/n_treatment*100:.2f}%")
print(f"  95% CI: [{ci_treatment[0]*100:.2f}%, {ci_treatment[1]*100:.2f}%]")
print(f"\nStatistical test:")
print(f"  Z-statistic: {z_stat:.4f}")
print(f"  P-value: {p_value:.6f}")
print(f"  Significant at α=0.05: {'Yes ✓' if p_value < 0.05 else 'No ✗'}")

---
## §5 · Bayesian A/B Testing

In [ ]:
# Bayesian A/B testing with Beta-Binomial model
# Prior: Beta(1, 1) = Uniform
alpha_prior = 1
beta_prior = 1

# Posterior parameters
alpha_control = alpha_prior + x_control
beta_control = beta_prior + (n_control - x_control)
alpha_treatment = alpha_prior + x_treatment
beta_treatment = beta_prior + (n_treatment - x_treatment)

# Sample from posteriors
n_samples = 100000
samples_control = np.random.beta(alpha_control, beta_control, n_samples)
samples_treatment = np.random.beta(alpha_treatment, beta_treatment, n_samples)

# Probability that treatment is better
prob_better = (samples_treatment > samples_control).mean()

print("=" * 70)
print("BAYESIAN A/B TEST RESULTS")
print("=" * 70)
print(f"\nPosterior distributions:")
print(f"  Control: Beta({alpha_control}, {beta_control})")
print(f"  Treatment: Beta({alpha_treatment}, {beta_treatment})")
print(f"\nP(Treatment > Control) = {prob_better:.4f}")
print(f"\nExpected lift: {(samples_treatment - samples_control).mean()*100:.2f}%")
print(f"95% credible interval: [{np.percentile(samples_treatment - samples_control, 2.5)*100:.2f}%, {np.percentile(samples_treatment - samples_control, 97.5)*100:.2f}%]")

---
## §6 · Minimum Detectable Effect

In [ ]:
# Power analysis for sample size calculation
# Parameters
effect_size = 0.03  # 3% lift
alpha = 0.05
power = 0.80

# Calculate required sample size
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

print("=" * 70)
print("POWER ANALYSIS")
print("=" * 70)
print(f"\nParameters:")
print(f"  Minimum detectable effect: {effect_size*100}%")
print(f"  Significance level (α): {alpha}")
print(f"  Power (1-β): {power}")
print(f"\nRequired sample size per group: {int(sample_size):,}")
print(f"Total users needed: {int(sample_size*2):,}")
print(f"\n💡 With {N_USERS:,} users, we have {N_USERS/sample_size/2:.1f}x the required sample size")

---
## §7 · Visualization

In [ ]:
# Visualize A/B test results
fig = make_subplots(rows=1, cols=2, subplot_titles=['Conversion Rates', 'Posterior Distributions'])

# Conversion rates
fig.add_trace(go.Bar(
    x=['Control', 'Treatment'],
    y=[x_control/n_control*100, x_treatment/n_treatment*100],
    error_y=dict(
        type='data',
        array=[(ci_control[1]-ci_control[0])/2*100, (ci_treatment[1]-ci_treatment[0])/2*100],
        visible=True
    ),
    marker_color=['#636EFA', '#EF553B']
), row=1, col=1)

# Posterior distributions
x_range = np.linspace(0.10, 0.18, 1000)
fig.add_trace(go.Scatter(
    x=x_range,
    y=stats.beta.pdf(x_range, alpha_control, beta_control),
    name='Control',
    fill='tozeroy',
    opacity=0.6
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_range,
    y=stats.beta.pdf(x_range, alpha_treatment, beta_treatment),
    name='Treatment',
    fill='tozeroy',
    opacity=0.6
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Clear separation between control and treatment")
print("   - High probability that treatment is better")
print("   - Sufficient sample size for confident decision")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Product Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | UI changes, feature releases, pricing experiments |
> | **Sample size** | Calculate upfront using power analysis |
> | **Statistical method** | Frequentist for compliance, Bayesian for decisions |
> | **Duration** | Run for full business cycles (min 1-2 weeks) |
> | **Decision rule** | Pre-define success criteria before starting |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Run A/B tests for all UI changes**:
>    ```
>    Design → Implement → Test → Analyze → Rollout
>    ```
>
> 2. **Common experiment types**:
>    | Experiment | Metric | Typical Lift |
>    |------------|--------|--------------|
>    | Button color/position | CTR | 2-5% |
>    | Simplified flow | Completion rate | 5-15% |
>    | Pricing changes | Revenue | 1-3% |
>    | Onboarding steps | Activation | 10-20% |
>
> 3. **Avoid common pitfalls**:
>    - Don't stop tests early (peeking problem)
>    - Don't run too many tests simultaneously
>    - Account for multiple comparisons
>
> ### 🔑 Key Takeaways
>
> 1. **Always calculate sample size** before starting an experiment.
> 2. **Use both frequentist and Bayesian** approaches for robust decisions.
> 3. **Visualize results** for stakeholder communication.
> 4. **Run tests for full business cycles** to capture weekly patterns.
> 5. **Document everything** for future reference and compliance.